In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import verde as vd
import choclo
import harmonica as hm
import pandas as pd
from scipy.interpolate import griddata
constant = choclo.constants.VACUUM_MAGNETIC_PERMEABILITY / (4 * np.pi)
import pathlib
import glob
import os
from stl import mesh
from pathlib import Path
import zipfile
import tempfile

import sys
sys.path.append("..")

import multipoles as mt
import micromag as mg
from micromag import MICROMETER_TO_METER, TESLA_TO_NANOTESLA
from tetrahedron_forward import *

In [2]:
def plot_maps(Bx_final, By_final, Bz_final, coordinates, height, title):
    # Cria as variáveis x e y
    x = coordinates[0] * 1.0e6
    y = coordinates[1] * 1.0e6
    cmap='seismic'
    levels=100
    
    # c = np.mean([X, Y, Z], axis=1)
    # height = np.round(sensor_sample_distance-c[2],3)
    # subplot 1x3
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    scale = np.max(abs(Bz_final.ravel()))
    scale *= 1.0e9
    
    # Plot Bx_final
    ax = axes[0]
    contour = ax.contourf(x, y, Bx_final*1.0e9, cmap=cmap, levels=levels, vmax=scale, vmin=-scale)
    # contour = ax.scatter(x, y, c=Bz_1d*1.0e9, cmap=cmap, s=2.5)# vmax=scale, vmin=-scale)
    fig.colorbar(contour, ax=ax, label='nT')
    # ax.scatter(X, Y, color='k', alpha=0.2, s=0.15)
    ax.set_xlabel(r'X $\mu m$')
    ax.set_ylabel(r'Y $\mu m$')
    ax.set_title(f'$B_x$ at height = {height} $\mu m$')
    
    # Plot By_final
    ax = axes[1]
    contour = ax.contourf(x, y, By_final*1.0e9, cmap=cmap, levels=levels, vmax=scale, vmin=-scale)
    # contour = ax.scatter(x, y, c=Bz_final*1.0e9, cmap=cmap, s=2.5)#, vmax=scale, vmin=-scale)
    fig.colorbar(contour, ax=ax, label='nT')
    # ax.scatter(x, y, color='k', alpha=0.2, s=0.15)
    ax.set_xlabel(r'X $\mu m$')
    ax.set_ylabel(r'Y $\mu m$')
    ax.set_title(f'$B_y$ at height = {height} $\mu m$')
    
    # Plot Bz_final
    ax = axes[2]
    # scale = np.max(abs(residual))
    contour = ax.contourf(x, y, Bz_final*1.0e9, cmap=cmap, levels=levels, vmax=scale, vmin=-scale)
    # contour = ax.scatter(x, y, c=residual, cmap=cmap, s=2.5)# vmax=scale, vmin=-scale)
    
    fig.colorbar(contour, ax=ax, label='nT')
    # ax.scatter(x, y, color='k', alpha=0.2, s=0.15)
    ax.set_xlabel(r'X $\mu m$')
    ax.set_ylabel(r'Y $\mu m$')
    ax.set_title(f'$B_z$ at height = {height} $\mu m$')

    fig.suptitle(title)
    # Ajusta o layout
    plt.tight_layout()
    # plt.savefig('irregular_86_comparison_slice_707.png', dpi=400)
    # Mostra o plot
    plt.show()

In [3]:
# Caminho para a pasta onde estão os arquivos
folder_path = pathlib.Path("../..") / "data/micromag_models/tecfiles"

# Lista todos os arquivos que terminam com "_mult.tec"
file_list = glob.glob(f"{folder_path}/*.tec.zip")

# Extrai apenas os nomes dos arquivos (sem o caminho completo)
file_names = [file.split("/")[-1] for file in file_list]

# Remove o sufixo '_mult.tec' somente na variável
body_samples_names = sorted([file.replace(".tec.zip", "") for file in file_names])

# Exibe os resultados
print(body_samples_names)

['eps0', 'eps1', 'eps10', 'eps11', 'eps12', 'eps13', 'eps14', 'eps15', 'eps16', 'eps17', 'eps18', 'eps19', 'eps2', 'eps3', 'eps4', 'eps5', 'eps6', 'eps7', 'eps8', 'eps9']


In [4]:
# Fixed sensor-sample distances (in micrometers, no conversion!)
heights = np.array([
    0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.35, 0.50, 0.75,
    1.00, 1.25, 1.50, 2.00, 3.00, 4.00, 5.00, 7.00, 10.00
])

# Directory containing STL files
stl_dir = pathlib.Path("../..") / "data/micromag_models/STL_ellipsoids"
stl_files = sorted(stl_dir.glob("*.stl"))

# Extract names (without extension)
names = [f.stem for f in stl_files]

# Loop over STL files
for name, stl_file in zip(names, stl_files):
    m = mesh.Mesh.from_file(str(stl_file))

    # Get maximum STL extent (in meters)
    STL_max_coord = max(
        np.max(abs(m.z)),
        np.max(abs(m.x)),
        np.max(abs(m.y))
    ) * MICROMETER_TO_METER

    # Loop over heights for this particle
    for height in heights:
        z0 = (height + (STL_max_coord * 1.05))
        bound = (STL_max_coord + z0 * 3)

        # Example output
        print(f"{name} | Height: {height} µm | Bound: {bound} µm")

eps0 | Height: 0.05 µm | Bound: 0.15000010375000156 µm
eps0 | Height: 0.075 µm | Bound: 0.22500010375000154 µm
eps0 | Height: 0.1 µm | Bound: 0.3000001037500016 µm
eps0 | Height: 0.15 µm | Bound: 0.4500001037500015 µm
eps0 | Height: 0.2 µm | Bound: 0.6000001037500017 µm
eps0 | Height: 0.25 µm | Bound: 0.7500001037500016 µm
eps0 | Height: 0.35 µm | Bound: 1.0500001037500013 µm
eps0 | Height: 0.5 µm | Bound: 1.5000001037500015 µm
eps0 | Height: 0.75 µm | Bound: 2.2500001037500015 µm
eps0 | Height: 1.0 µm | Bound: 3.0000001037500015 µm
eps0 | Height: 1.25 µm | Bound: 3.7500001037500015 µm
eps0 | Height: 1.5 µm | Bound: 4.5000001037500015 µm
eps0 | Height: 2.0 µm | Bound: 6.0000001037500015 µm
eps0 | Height: 3.0 µm | Bound: 9.000000103750002 µm
eps0 | Height: 4.0 µm | Bound: 12.000000103750002 µm
eps0 | Height: 5.0 µm | Bound: 15.000000103750002 µm
eps0 | Height: 7.0 µm | Bound: 21.000000103750004 µm
eps0 | Height: 10.0 µm | Bound: 30.00000010375 µm
eps1 | Height: 0.05 µm | Bound: 0.150000

In [5]:
output_dir = folder_path / "grids"
output_dir.mkdir(exist_ok=True)

In [6]:
# define this earlier in your code
output_dir = folder_path / "grids"
output_dir.mkdir(exist_ok=True)

for height in heights:
    for body_sample in body_samples_names:
        # Define output file path
        output_file = output_dir / f"grid_{body_sample}_{str(height)}.nc"
        
        # Skip if file already exists
        if output_file.exists():
            print(f"Skipping {output_file.name} (already exists)")
            continue

        stl_file = stl_dir / f"{body_sample}.stl"
        m = mesh.Mesh.from_file(str(stl_file))

        # STL max extent in micrometers (no conversion)
        STL_max_coord = max(
            np.max(abs(m.z)),
            np.max(abs(m.x)),
            np.max(abs(m.y))
        )
        STL_max_z_coord=np.max(abs(m.z))
        # Compute z0 and bound
        z0 = height + (STL_max_z_coord * 1.00)
        bound = STL_max_coord + z0 * 3


        
        # dir_file_path = folder_path
        # # read the file data
        # file_path =f'{body_sample}.tec'
        # file_path = dir_file_path / file_path
        # file_content = read_tecplot_file(file_path)

        dir_file_path = folder_path
        file_path = dir_file_path / f"{body_sample}.tec.zip"
        
        with zipfile.ZipFile(file_path) as z:
            with z.open(z.namelist()[0]) as f:
                with tempfile.NamedTemporaryFile(delete=False, suffix=".tec") as tmp:
                    tmp.write(f.read())
                    file_path = tmp.name
        
        file_content = read_tecplot_file(file_path)

        
        title, variables, N, E = read_tecplot_header(file_content)
        variables_values, sd_values, vertices_indexes = organize_data(file_content, N, E)
    
        # get the specific body index
        vertices_indexes_filtered = list()   
        for i, vertices_array in enumerate(vertices_indexes):
            if sd_values[i] == 101.0:
                vertices_indexes_filtered.append(np.asarray(vertices_array))
    
        # get nodes data
        X, Y, Z = variables_values['X'], variables_values['Y'], variables_values['Z']
        Mx, My, Mz = variables_values['Mx'], variables_values['My'], variables_values['Mz']
        # get tetrahedrons
        tetrahedrons, tetrahedrons_centroid = define_tetrahedrons(X, Y, Z, vertices_indexes_filtered)
        # sort tetrahedrons
        tetrahedrons_sorted = sort_tetrahedrons(tetrahedrons, tetrahedrons_centroid)
        # tetrahedrons magnetization
        Ms = 482476.91085597320
        tetrahedrons_mag = define_tetrahedrons_magnetizations_weighted(X, Y, Z, 
                                                                       Mx, My, Mz, 
                                                                       vertices_indexes_filtered, Ms=Ms)
        # get true magnetic moment of the grain
        true_magnetic_moment = calculate_grain_magnetic_moments(X, Y, Z, 
                                                                Mx, My, Mz, 
                                                                vertices_indexes_filtered, Ms=Ms)
            
        # generate coordinates grid
        c = np.mean([X, Y, Z], axis=1) # particles center
        x_windows_size = bound   
        y_windows_size = bound  
        spacing = (x_windows_size+y_windows_size)/50 # always make 51x51 grids
        
        
        sensor_sample_distance = z0 #np.max(Z)+height#c[2]+height  # µm  ############### AQUIIIIIIII ################
        coordinates = vd.grid_coordinates(
            region=[-x_windows_size+c[0], x_windows_size+c[0], -y_windows_size+c[1], y_windows_size+c[1]],  # µm
            spacing=spacing,  # µm
            extra_coords=sensor_sample_distance,
        )
        
        coordinates = np.asarray(coordinates)*1.0e-6
        shape = np.shape(coordinates[0])
    
        # calculate model
        Bx_final = np.zeros(len(coordinates[0].ravel()))
        By_final = np.zeros(len(coordinates[0].ravel()))
        Bz_final = np.zeros(len(coordinates[0].ravel()))
        calculate_Omega_f_and_Delta_W_f(tetrahedrons_sorted, tetrahedrons_mag, coordinates, Bx_final, By_final, Bz_final)
    
        Bx_final = np.reshape(Bx_final, shape)
        By_final = np.reshape(By_final, shape)
        Bz_final = np.reshape(Bz_final, shape)
    
        plot = False
        if plot:
            plot_maps(Bx_final, By_final, Bz_final, coordinates, height, title=body_sample)
            print()
        # create the xarray dataset
        data = vd.make_xarray_grid(coordinates*1.0e6, Bz_final*1.0e9, data_names=["bz"], dims=("y", "x"), extra_coords_names="z")
        data.x.attrs = {"units": "µm"}
        data.y.attrs = {"units": "µm"}
        data.bz.attrs = {"long_name": "bz", "units": "nT"}

        # add the other components to the dataset if needed
        other_components = False
        if other_components:
            data = data.assign(bx=(['y','x'],Bx_final*1.0e9))
            data.bx.attrs = {"long_name": "bx", "units": "nT"}
            data = data.assign(by=(['y','x'],By_final*1.0e9))
            data.by.attrs = {"long_name": "by", "units": "nT"}
            b111 = np.array([0, np.sqrt(2/3), np.sqrt(1/3)])
            dot_product = Bx_final*b111[0]+By_final*b111[1]+Bz_final*b111[2]
            data = data.assign(b111=(['y','x'],dot_product*1.0e9))
            data.b111.attrs = {"long_name": "b111", "units": "nT"}
        
        # add the sensor height and the particle's centroid and magnetic moment as extra information 
        data = data.assign(sensor_height=height)
        data = data.assign(centroid=c)
        data = data.assign(magnetic_moment=true_magnetic_moment)
        data.centroid.attrs = {"long_name": "centroid position", "units": r"µm"}
        data.sensor_height.attrs = {"long_name": "sensor height", "units": r"µm"}
        data.magnetic_moment.attrs = {"long_name": "magnetic moment", "units": r"Am²"}

        data = data.assign(body_name=body_sample)
        
        # save the xarray into netcdf files
        data.to_netcdf(output_dir/f"grid_{body_sample}_{str(height)}.nc")

Skipping grid_eps0_0.05.nc (already exists)
Skipping grid_eps1_0.05.nc (already exists)
Skipping grid_eps10_0.05.nc (already exists)
Skipping grid_eps11_0.05.nc (already exists)
Skipping grid_eps12_0.05.nc (already exists)
Skipping grid_eps13_0.05.nc (already exists)
Skipping grid_eps14_0.05.nc (already exists)
Skipping grid_eps15_0.05.nc (already exists)
Skipping grid_eps16_0.05.nc (already exists)
Skipping grid_eps17_0.05.nc (already exists)
Skipping grid_eps18_0.05.nc (already exists)
Skipping grid_eps19_0.05.nc (already exists)
Skipping grid_eps2_0.05.nc (already exists)
Skipping grid_eps3_0.05.nc (already exists)
Skipping grid_eps4_0.05.nc (already exists)
Skipping grid_eps5_0.05.nc (already exists)
Skipping grid_eps6_0.05.nc (already exists)
Skipping grid_eps7_0.05.nc (already exists)
Skipping grid_eps8_0.05.nc (already exists)
Skipping grid_eps9_0.05.nc (already exists)
Skipping grid_eps0_0.075.nc (already exists)
Skipping grid_eps1_0.075.nc (already exists)
Skipping grid_eps10_